# Set YAML, agents and outputs

## Goal
Paths are relative to YAML. One controlled reservoir needs one agent; passive outlets have no LLM. Change run_id and output for a new experiment. A model profile is required only for live LLM mode.

## Setup
Python 3.10+ with repository dependencies. All cells use local files; no provider calls.

In [1]:
from pathlib import Path
import json, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/hydrocnhs').is_dir())
sys.path.insert(0, str(ROOT/'src'))
from llm_reservoir_agent.workflow import load_settings, run_settings, verify_settings_output
from llm_reservoir_agent.verification import verify_run
from unittest.mock import patch
def ensure_offline(config):
    settings = load_settings(ROOT/config)
    assert settings.report['planned_llm_calls'] == 0
    output = Path(settings.report['output'])
    if not output.exists():
        with patch('urllib.request.urlopen', side_effect=AssertionError('network forbidden')):
            run_settings(settings)
    return settings, verify_settings_output(settings)
print('Checkout:', ROOT.name)


Checkout: clean-upgrade-99ea43e42a


## Steps

### Inspect commented YAML

In [2]:
print((ROOT/'examples/fe_walter/dkp/experiment.tutorial.yaml').read_text())

# FE Walter V3.7 dkp: original prompt builders, synthetic replies, zero model calls.
schema_version: llm-reservoir-experiment.v1
case: ../../../cases/fe_walter_reference/case.json # Frozen hydrology sample; V3.7 physics.
run_id: fe-walter-tutorial-dkp        # Change with output before a new run.
output: ../../../runs/fe-walter-tutorial-dkp
max_calls: 0                             # This example cannot call a provider.
auditors:
  release_limits:
    type: release_bounds
    version: "1"
agents:
  fe_operator:
    reservoir: FEWalter                  # Reservoir ID in case.json.
    mode: offline                        # Native prompt/audit run on fixture replies.
    strategy: reference_v3_7             # Generic strategy selector; optional vendor plugin.
    reference:
      profile: fe_walter                 # This reference profile is FE-specific.
      method: dkp                   # Original zero-shot or DKP policy.
      system_prompt: ../../../prompts/fe_walter_reference/dkp_sy

### Validate without a provider

In [3]:
for config in ['examples/fe_walter/benchmark/experiment.yaml','examples/fe_walter/zero_shot/experiment.tutorial.yaml','examples/fe_walter/dkp/experiment.tutorial.yaml']:
    plan=load_settings(ROOT/config).report
    print(config, {k:plan[k] for k in ['days','agent_count','agent_strategies','planned_llm_calls']})
live=load_settings(ROOT/'examples/fe_walter/dkp/experiment.llm.yaml')
print('Live template temperature:', live.data['models']['reservoir_model']['temperature'])
print('Live readiness:', live.report['live_ready'])

examples/fe_walter/benchmark/experiment.yaml {'days': 3, 'agent_count': 1, 'agent_strategies': {'fe_operator': 'benchmark'}, 'planned_llm_calls': 0}
examples/fe_walter/zero_shot/experiment.tutorial.yaml {'days': 3, 'agent_count': 1, 'agent_strategies': {'fe_operator': 'reference_v3_7'}, 'planned_llm_calls': 0}
examples/fe_walter/dkp/experiment.tutorial.yaml {'days': 3, 'agent_count': 1, 'agent_strategies': {'fe_operator': 'reference_v3_7'}, 'planned_llm_calls': 0}
Live template temperature: 0.7
Live readiness: False


## Checks
Review the printed inputs, counts, source labels and residuals. Synthetic replies validate interfaces only.

## Next steps
check never calls the provider. Actual live execution needs confirmed case readiness, a named installed model, a budget and explicit --allow-network. This tutorial stops at offline verification.